
# Trader Performance vs Market Sentiment Analysis

This notebook analyzes how **Bitcoin market sentiment (Fear/Greed)** affects **trader behavior and profitability**.

Datasets used:
- Fear & Greed Index
- Historical Hyperliquid trader data

Goals:
1. Clean and prepare the datasets
2. Merge sentiment data with trading data
3. Create trader performance metrics
4. Analyze behavior under Fear vs Greed sentiment
5. Extract actionable insights


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")


## Load Datasets

In [ ]:

sentiment = pd.read_csv("../data/fear_greed_index.csv")
trades = pd.read_csv("../data/historical_data.csv")

print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)


## Data Exploration

In [ ]:

sentiment.head()


In [ ]:

trades.head()


## Check Missing Values

In [ ]:

print("Sentiment Missing Values:")
print(sentiment.isnull().sum())

print("\nTrades Missing Values:")
print(trades.isnull().sum())


## Convert Timestamp Columns

In [ ]:

trades['Timestamp'] = pd.to_datetime(trades['Timestamp'], unit='ms')
sentiment['date'] = pd.to_datetime(sentiment['date'])

trades['date'] = trades['Timestamp'].dt.date
sentiment['date_only'] = sentiment['date'].dt.date


## Merge Sentiment and Trader Data

In [ ]:

merged = pd.merge(
    trades,
    sentiment[['date_only','classification','value']],
    left_on='date',
    right_on='date_only',
    how='left'
)

merged.head()


## Feature Engineering

In [ ]:

daily_pnl = merged.groupby(['Account','date'])['Closed PnL'].sum().reset_index()
daily_pnl.head()


In [ ]:

merged['win'] = merged['Closed PnL'] > 0
win_rate = merged.groupby('Account')['win'].mean()
win_rate.head()


In [ ]:

avg_trade_size = merged.groupby('Account')['Size USD'].mean()
avg_trade_size.head()


## Trades per Day

In [ ]:

trades_per_day = merged.groupby('date').size()

plt.figure(figsize=(10,4))
trades_per_day.plot()
plt.title("Number of Trades Per Day")
plt.show()


## Long vs Short Trades

In [ ]:

sns.countplot(x='Side', data=merged)
plt.title("Long vs Short Trades")
plt.show()


## PnL vs Market Sentiment

In [ ]:

pnl_sentiment = merged.groupby('classification')['Closed PnL'].mean()

sns.barplot(x=pnl_sentiment.index, y=pnl_sentiment.values)
plt.title("Average PnL during Fear vs Greed")
plt.show()


## Trade Activity vs Sentiment

In [ ]:

trade_freq = merged.groupby('classification').size()

sns.barplot(x=trade_freq.index, y=trade_freq.values)
plt.title("Trading Activity by Market Sentiment")
plt.show()


## Position Size vs Sentiment

In [ ]:

sns.boxplot(x='classification', y='Size USD', data=merged)
plt.title("Position Size vs Sentiment")
plt.show()


## Trader Segmentation

In [ ]:

trade_counts = merged['Account'].value_counts()
median_trades = trade_counts.median()

merged['freq_segment'] = merged['Account'].map(
    lambda x: "High Frequency" if trade_counts[x] > median_trades else "Low Frequency"
)


In [ ]:

pnl_trader = merged.groupby('Account')['Closed PnL'].sum()
median_pnl = pnl_trader.median()

merged['profit_segment'] = merged['Account'].map(
    lambda x: "Consistent Winner" if pnl_trader[x] > median_pnl else "Inconsistent"
)



## Key Insights

1. Trading activity tends to increase during **Greed** sentiment periods.
2. Larger trade sizes appear during strong bullish sentiment.
3. High-frequency traders tend to maintain more consistent performance.



## Strategy Recommendations

1. Reduce position sizes during **Fear markets** to manage downside risk.
2. Increase trading activity during **Greed markets** to capture momentum.
3. Use sentiment signals to adjust position sizing dynamically.
